# Extra Trees profile features for the neural-network bottom third

This notebook loads the 27 participants in the bottom third of the final CSP–MLP participant-accuracy distribution and retrieves their cleaned profile information. The displayed dataframe contains exactly the 15 factors selected by the Extra Trees held-out permutation-importance analysis in `model.ipynb`. Participant ID is retained only as the row index.

In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_data_root(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed'
        if (candidate / 'Perfomances_cleaned.csv').is_file():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Perfomances_cleaned.csv.')


DATA_ROOT = find_data_root()
MODEL_OUTPUT_ROOT = DATA_ROOT / 'processed' / 'csp_neural_net_physio_27'
PROFILE_PATH = DATA_ROOT / 'processed' / 'Perfomances_cleaned.csv'
BOTTOM_THIRD_PATH = MODEL_OUTPUT_ROOT / 'all_79_bottom_third_participants.csv'
OUTPUT_PATH = MODEL_OUTPUT_ROOT / 'bottom_third_extra_trees_profile_features.csv'

print('Profiles:', PROFILE_PATH)
print('Neural-network bottom third:', BOTTOM_THIRD_PATH)

Profiles: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/Perfomances_cleaned.csv
Neural-network bottom third: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/csp_neural_net_physio_27/all_79_bottom_third_participants.csv


## Extra Trees-selected profile factors

These are the 15 original profile variables with the highest mean increase in held-out RMSE when individually shuffled for the Extra Trees model. The ordering follows the permutation-importance ranking saved by the analysis in `model.ipynb`.

In [6]:
EXTRA_TREES_SELECTED_FEATURES = [
    'PRE_Motivation',
    'PRE_Stim_normal',
    'Level of study',
    'Vision',
    'Q4',
    'Manual activity',
    'O',
    'Vision_assistance',
    'Meditation practice',
    'A',
    'verbal',
    'PRE_Mood',
    'intuitive',
    'sequential',
    'visual',
]

bottom_third = pd.read_csv(BOTTOM_THIRD_PATH)
profiles = pd.read_csv(PROFILE_PATH, sep=';')

if bottom_third['participant'].duplicated().any():
    raise ValueError('The neural-network bottom-third file contains duplicate participants.')
if profiles['SUJ_ID'].duplicated().any():
    raise ValueError('The cleaned profile file contains duplicate participant IDs.')

missing_features = sorted(
    set(EXTRA_TREES_SELECTED_FEATURES) - set(profiles.columns)
)
if missing_features:
    raise KeyError(f'Missing Extra Trees-selected profile fields: {missing_features}')

ranked_participants = bottom_third['participant'].astype(str).tolist()
profile_lookup = profiles.assign(
    SUJ_ID=profiles['SUJ_ID'].astype(str)
).set_index('SUJ_ID')
unmatched_participants = sorted(set(ranked_participants) - set(profile_lookup.index))
if unmatched_participants:
    raise ValueError(f'No cleaned profile found for: {unmatched_participants}')

# Participant is the index; the dataframe columns are only the selected factors.
bottom_third_selected_profiles = profile_lookup.loc[
    ranked_participants, EXTRA_TREES_SELECTED_FEATURES
].copy()
bottom_third_selected_profiles.index.name = 'Participant'

assert len(bottom_third_selected_profiles) == 27
assert bottom_third_selected_profiles.index.is_unique
assert bottom_third_selected_profiles.columns.tolist() == EXTRA_TREES_SELECTED_FEATURES

bottom_third_selected_profiles.to_csv(OUTPUT_PATH)
print(
    f'Loaded {len(bottom_third_selected_profiles)} participants with '
    f'{len(EXTRA_TREES_SELECTED_FEATURES)} selected profile features.'
)
print('Saved:', OUTPUT_PATH)
display(bottom_third_selected_profiles)

Loaded 27 participants with 15 selected profile features.
Saved: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/csp_neural_net_physio_27/bottom_third_extra_trees_profile_features.csv


,PRE_Motivation,PRE_Stim_normal,Level of study,Vision,Q4,Manual activity,O,Vision_assistance,Meditation practice,A,verbal,PRE_Mood,intuitive,sequential,visual
Participant,,,,,,,,,,,,,,,
A43,100,2,3,1,5.0,5,3.0,0,1.0,2.0,2.0,96,4.0,6.0,9.0
A34,100,2,3,1,8.0,2,8.0,0,NaN,3.0,2.0,44,4.0,6.0,9.0
A27,80,2,3,2,5.0,4,5.0,1,3.0,3.0,4.0,68,2.0,4.0,7.0
B77,80,1,8,2,6.0,2,8.0,2,1.0,1.0,8.0,56,7.0,9.0,3.0
B67,100,2,8,2,5.0,3,6.0,2,1.0,1.0,4.0,80,7.0,5.0,7.0
B71,100,1,5,1,6.0,3,4.0,0,1.0,2.0,5.0,60,2.0,6.0,6.0
C82,20,1,5,1,1.0,4,3.0,0,2.0,1.0,1.0,52,8.0,3.0,10.0
B68,100,2,3,1,4.0,5,3.0,0,1.0,1.0,3.0,68,2.0,4.0,8.0
A11,80,2,3,2,6.0,4,8.0,2,1.0,3.0,2.0,80,3.0,7.0,9.0


In [7]:
# Retrieve the same selected profile features for the 27 participants with
# the lowest strict mean performance accuracy across Runs 3, 4, 5, and 6.
PERFORMANCE_RUN_COLUMNS = [
    'Perf_RUN_3', 'Perf_RUN_4', 'Perf_RUN_5', 'Perf_RUN_6'
]
missing_run_columns = sorted(
    set(PERFORMANCE_RUN_COLUMNS) - set(profiles.columns)
)
if missing_run_columns:
    raise KeyError(f'Missing performance columns: {missing_run_columns}')

performance_ranking = profiles[['SUJ_ID', *PERFORMANCE_RUN_COLUMNS]].copy()
performance_ranking['aggregate_mean_performance_accuracy'] = (
    performance_ranking[PERFORMANCE_RUN_COLUMNS].mean(axis=1, skipna=False)
)
performance_ranking = performance_ranking.dropna(
    subset=['aggregate_mean_performance_accuracy']
)
eligible_performance_count = len(performance_ranking)
performance_ranking = (
    performance_ranking
    .sort_values(
        ['aggregate_mean_performance_accuracy', 'SUJ_ID'],
        ascending=[True, True],
        kind='mergesort',
    )
    .head(27)
)
bottom_27_performance_participants = (
    performance_ranking['SUJ_ID'].astype(str).tolist()
)

# Participant is the index; the dataframe columns are only the selected factors.
bottom_27_performance_selected_profiles = profile_lookup.loc[
    bottom_27_performance_participants, EXTRA_TREES_SELECTED_FEATURES
].copy()
bottom_27_performance_selected_profiles.index.name = 'Participant'

assert len(bottom_27_performance_selected_profiles) == 27
assert bottom_27_performance_selected_profiles.index.is_unique
assert (
    bottom_27_performance_selected_profiles.columns.tolist()
    == EXTRA_TREES_SELECTED_FEATURES
)

PERFORMANCE_OUTPUT_PATH = (
    MODEL_OUTPUT_ROOT
    / 'bottom_27_performance_extra_trees_profile_features.csv'
)
bottom_27_performance_selected_profiles.to_csv(PERFORMANCE_OUTPUT_PATH)
performance_cutoff = performance_ranking[
    'aggregate_mean_performance_accuracy'
].max()
print(
    f'Loaded the bottom 27 of {eligible_performance_count} eligible '
    f'participants; aggregate-accuracy cutoff = {performance_cutoff:.3f}%.'
)
print('Saved:', PERFORMANCE_OUTPUT_PATH)
display(bottom_27_performance_selected_profiles)

Loaded the bottom 27 of 86 eligible participants; aggregate-accuracy cutoff = 51.875%.
Saved: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/csp_neural_net_physio_27/bottom_27_performance_extra_trees_profile_features.csv


,PRE_Motivation,PRE_Stim_normal,Level of study,Vision,Q4,Manual activity,O,Vision_assistance,Meditation practice,A,verbal,PRE_Mood,intuitive,sequential,visual
Participant,,,,,,,,,,,,,,,
A30,100,2,3,1,7.0,4,7.0,0,NaN,5.0,3.0,56,6.0,5.0,8.0
B72,100,2,8,1,6.0,4,9.0,0,1.0,4.0,8.0,64,2.0,8.0,3.0
C83,100,1,3,1,NaN,2,NaN,0,1.0,NaN,NaN,92,NaN,NaN,NaN
A17,100,2,3,1,6.0,3,6.0,0,1.0,4.0,1.0,84,3.0,4.0,10.0
A27,80,2,3,2,5.0,4,5.0,1,3.0,3.0,4.0,68,2.0,4.0,7.0
A47,80,2,3,1,5.0,2,3.0,0,1.0,3.0,6.0,56,7.0,3.0,5.0
A53,100,2,3,1,8.0,2,7.0,0,1.0,4.0,1.0,92,4.0,6.0,10.0
A37,80,2,3,2,7.0,4,6.0,0,1.0,3.0,2.0,72,8.0,5.0,9.0
A54,80,2,3,1,5.0,4,6.0,0,1.0,5.0,4.0,68,5.0,2.0,7.0


In [8]:
# List participants appearing in both bottom-27 groups.
performance_bottom_set = set(
    bottom_27_performance_selected_profiles.index.astype(str)
)
shared_participant_ids = [
    participant
    for participant in bottom_third_selected_profiles.index.astype(str)
    if participant in performance_bottom_set
]
shared_bottom_27_participants = pd.DataFrame({
    'Participant': shared_participant_ids
})

SHARED_OUTPUT_PATH = (
    MODEL_OUTPUT_ROOT / 'shared_bottom_27_participants.csv'
)
shared_bottom_27_participants.to_csv(SHARED_OUTPUT_PATH, index=False)
print(
    f'Shared participants: {len(shared_bottom_27_participants)} of 27 '
    f'({len(shared_bottom_27_participants) / 27:.1%})'
)
print('Saved:', SHARED_OUTPUT_PATH)
display(shared_bottom_27_participants)

Shared participants: 18 of 27 (66.7%)
Saved: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/csp_neural_net_physio_27/shared_bottom_27_participants.csv


,Participant
0,A34
1,A27
2,B67
3,B71
4,C82
5,A11
6,B64
7,B72
8,A30
9,A42
